In [6]:
import pandas as pd
import yfinance as yf
import time

df = pd.read_csv('../data/processed/ipo_clean.csv')
df['listing_date'] = pd.to_datetime(df['listing_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   company_name       397 non-null    str           
 1   nse_symbol         397 non-null    str           
 2   listing_date       397 non-null    datetime64[us]
 3   issue_price        397 non-null    int64         
 4   listing_price      397 non-null    float64       
 5   issue_size_crores  397 non-null    float64       
 6   QIB                397 non-null    float64       
 7   HNI                397 non-null    float64       
 8   RII                397 non-null    float64       
 9   Total              397 non-null    float64       
dtypes: datetime64[us](1), float64(6), int64(1), str(2)
memory usage: 31.1 KB


In [7]:
DATA_CUTOFF_DATE = '2026-06-30'

all_price_data = [] #to collect result
failed_tickers = [] 

for idx,row in df.iterrows(): #access every row
    symbol = row['nse_symbol']
    listing_date_str = row['listing_date'].strftime('%Y-%m-%d')
    yf_symbol = symbol + '.NS'

    try:
        ticker_obj = yf.Ticker(yf_symbol)
        hist = ticker_obj.history(start = listing_date_str, end = DATA_CUTOFF_DATE, auto_adjust=True) #auto-adjusts for any split or dividends

        if hist.empty:
            failed_tickers.append(symbol)
            continue

        hist = hist.reset_index() #so that date becomes a column and not index
        hist['nse_symbol'] = symbol
        all_price_data.append(hist[['nse_symbol','Date', 'Close']])

    except Exception as e:
        failed_tickers.append(symbol)
        print(f"Failed {symbol} - {e}")

    time.sleep(0.3) #to give a break btw each API call

print(f"Success = {len(all_price_data)}/{len(df)}")
print(f"Failure = {len(failed_tickers)}/{len(df)}")       

Success = 397/397
Failure = 0/397


In [15]:
price_history = pd.concat(all_price_data, ignore_index=True)
print(price_history.shape)
price_history.head(15)

(305199, 3)


,nse_symbol,Date,Close
0,GKSL,2025-12-30 00:00:00+05:30,104.540001
1,GKSL,2025-12-31 00:00:00+05:30,102.970001
2,GKSL,2026-01-01 00:00:00+05:30,103.440002
3,GKSL,2026-01-02 00:00:00+05:30,102.709999
4,GKSL,2026-01-05 00:00:00+05:30,102.400002
5,GKSL,2026-01-06 00:00:00+05:30,100.889999
6,GKSL,2026-01-07 00:00:00+05:30,103.709999
7,GKSL,2026-01-08 00:00:00+05:30,103.019997
8,GKSL,2026-01-09 00:00:00+05:30,102.169998
9,GKSL,2026-01-12 00:00:00+05:30,102.680000


In [17]:
# How many unique companies do we actually have data for?
print(f"Unique companies: {price_history['nse_symbol'].nunique()}")
# Any nulls in Close prices?
print(f"\nNull Close prices: {price_history['Close'].isnull().sum()}")
# Date range overview
print(f"\nOverall date range: {price_history['Date'].min()} to {price_history['Date'].max()}")

Unique companies: 397

Null Close prices: 0

Overall date range: 2018-01-22 00:00:00+05:30 to 2026-06-29 00:00:00+05:30


In [18]:
price_history.to_csv('../data/processed/price_history.csv', index = False)
print(price_history.shape)

(305199, 3)


In [20]:
nifty = yf.Ticker('^NSEI')
nifty_hist = nifty.history(start = '2018-01-01', end = DATA_CUTOFF_DATE , auto_adjust=True)
nifty_hist = nifty_hist.reset_index()[['Date' , 'Close']]
nifty_hist = nifty_hist.rename(columns = {'Close':'nifty_close'})

print(nifty_hist.shape)
nifty_hist.head(15)

(2090, 2)


,Date,nifty_close
0,2018-01-02 00:00:00+05:30,10442.200195
1,2018-01-03 00:00:00+05:30,10443.200195
2,2018-01-04 00:00:00+05:30,10504.799805
3,2018-01-05 00:00:00+05:30,10558.849609
4,2018-01-08 00:00:00+05:30,10623.599609
5,2018-01-09 00:00:00+05:30,10637.000000
6,2018-01-10 00:00:00+05:30,10632.200195
7,2018-01-11 00:00:00+05:30,10651.200195
8,2018-01-12 00:00:00+05:30,10681.250000
9,2018-01-15 00:00:00+05:30,10741.549805


In [21]:
nifty_hist.to_csv('../data/processed/nifty50_history.csv', index=False)

## Step 2 Summary: Price History Enrichment

Pulled daily closing prices (split/bonus-adjusted via `auto_adjust=True`) for all 
397 matched NSE tickers using `yfinance`'s `Ticker().history()` method. Each 
company's pull spans from its listing date to a fixed cutoff of **2026-06-30**, 
chosen instead of a dynamic "today" to keep results reproducible on re-runs.

Also pulled Nifty 50 (`^NSEI`) daily closes from 2018-01-01 to the same cutoff, 
to serve as the market benchmark for all relative return calculations in Step 3.

**Results:**
- 397/397 tickers resolved successfully, no failures
- 3,05,199 total daily price rows across all companies
- Zero nulls in Close prices across the full dataset
  
**Design decisions:**
- Used `Ticker().history()` rather than `download()` — returns flat (non-MultiIndex) 
  columns, simpler to work with in a per-company loop
- Used `auto_adjust=True` so returned "Close" prices are already split/bonus-adjusted, 
  preventing corporate actions from distorting return calculations
- Pulled full daily history per company in a single API call (listing date → cutoff), 
  rather than separate calls per horizon (6mo/1yr/2yr/3yr) — snapshot values for 
  each horizon will be extracted from this single time series in Step 3
- Added a 0.3s delay between requests to avoid yfinance rate-limiting
  
**Files saved:**
- `data/processed/price_history.csv` — daily closes per company
- `data/processed/nifty50_history.csv` — daily closes for the Nifty 50 benchmark